In [137]:
import pandas as pd

In [138]:
df = pd.read_csv('movie.csv')

In [139]:
df.shape

(10000, 11)

In [140]:
df.isnull().sum()

original_title      0
overview            0
genres              0
tagline             0
keywords            0
directors           0
cast                0
poster_path         0
status              0
spoken_languages    0
averageRating       0
dtype: int64

In [141]:
df = df.drop_duplicates()

In [142]:
df = df.drop(['poster_path','tagline'],axis=1)

In [143]:
df.head()

,original_title,overview,genres,keywords,directors,cast,status,spoken_languages,averageRating
0,Mortal Kombat Legends: Scorpion's Revenge,After the vicious slaughter of his family by s...,"Animation, Action, Fantasy","magic, ninja fighter, revenge, sorcerer, tourn...",Ethan Spaulding,"Patrick Seitz, Jordan Rodrigues, Jennifer Carp...",Released,English,7.4
1,Defenders of Life,"In the hills of Costa Rica, Doña Carmen strugg...",Drama,"rape, tradition, indigenous, pregnancy, pregna...",Dana Ziyasheva,"Beatriz Brenes, Arman Darbo, Eylin Esther Jime...",Released,Spanish,7.4
2,新・破裏拳ポリマー,Takeshi is a young man who receives the Polyma...,"Action, Animation, Science Fiction","japan, superhero",Akiyuki Shinbō,"Takeshi Aono, Ryotaro Okiayu, Yuko Miyamura, A...",Released,Japanese,5.9
3,The Pianist,The true story of pianist Władysław Szpilman's...,"Drama, War","concert, nazi, resistance, warsaw ghetto, poli...",Roman Polanski,"Adrien Brody, Thomas Kretschmann, Frank Finlay...",Released,"English, Polish, German, Russian",8.5
4,Suzi Q,Story of trailblazing American rock singer-son...,"Documentary, Music","rock music, female rocker",Liam Firmager,"Suzi Quatro, Alice Cooper, Debbie Harry, Tina ...",Released,English,7.2


In [144]:
df['tags'] = (
    df['genres'] + ' ' +
    df['keywords'] + ' ' +
    df['overview'] + ' ' +
    df['cast'] + ' ' +
    df['directors']
)

In [145]:
df = df.drop(['genres', 'keywords', 'overview','cast','directors'],axis=1)

In [146]:
df.head()

,original_title,status,spoken_languages,averageRating,tags
0,Mortal Kombat Legends: Scorpion's Revenge,Released,English,7.4,"Animation, Action, Fantasy magic, ninja fighte..."
1,Defenders of Life,Released,Spanish,7.4,"Drama rape, tradition, indigenous, pregnancy, ..."
2,新・破裏拳ポリマー,Released,Japanese,5.9,"Action, Animation, Science Fiction japan, supe..."
3,The Pianist,Released,"English, Polish, German, Russian",8.5,"Drama, War concert, nazi, resistance, warsaw g..."
4,Suzi Q,Released,English,7.2,"Documentary, Music rock music, female rocker S..."


In [147]:
import re 

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]",'',text)
    text = re.sub(r"\s+",' ', text).strip()
    return text
df['tags'] = df['tags'].apply(clean_text)

In [148]:
df.head()

,original_title,status,spoken_languages,averageRating,tags
0,Mortal Kombat Legends: Scorpion's Revenge,Released,English,7.4,animation action fantasy magic ninja fighter r...
1,Defenders of Life,Released,Spanish,7.4,drama rape tradition indigenous pregnancy preg...
2,新・破裏拳ポリマー,Released,Japanese,5.9,action animation science fiction japan superhe...
3,The Pianist,Released,"English, Polish, German, Russian",8.5,drama war concert nazi resistance warsaw ghett...
4,Suzi Q,Released,English,7.2,documentary music rock music female rocker sto...


In [149]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embedding = model.encode(
    df['tags'].tolist(),
    show_progress_bar=True
)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

In [150]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(embedding)

In [151]:
import numpy as np
def recommend(movie_name, n=10):

    # Check whether movie exists
    matches = df[
        df['original_title'].str.lower() == movie_name.lower()
    ]

    if matches.empty:
        print("Movie not found.")
        return

    movie_index = matches.index[0]

    # Calculate similarity
    similarity_scores = cosine_similarity(
        embedding[movie_index].reshape(1, -1),
        embedding
    )[0]

    # Get sorted indices
    similar_movies = np.argsort(
        similarity_scores
    )[::-1]

    print(f"\nRecommendations for: {df.loc[movie_index, 'original_title']}")
    print("-" * 50)

    count = 0

    for index in similar_movies:

        # Don't recommend the same movie
        if index == movie_index:
            continue

        print(
            f"{count + 1}. "
            f"{df.loc[index, 'original_title']} "
            f"(Similarity: {similarity_scores[index]:.3f})"
        )

        count += 1

        if count == n:
            break

In [152]:
recommend("The Pianist", 5)


Recommendations for: The Pianist
--------------------------------------------------
1. Miasto 44 (Similarity: 0.745)
2. Love Gets a Room (Similarity: 0.728)
3. Warszawa: miasto podzielone (Similarity: 0.712)
4. Phoenix (Similarity: 0.672)
5. Správa (Similarity: 0.661)
